In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

---
## 1. Load & Clean Data

In [ ]:
train        = pd.read_csv('train.csv',           parse_dates=['date'])
test         = pd.read_csv('test.csv',            parse_dates=['date'])
stores       = pd.read_csv('stores.csv')
oil          = pd.read_csv('oil.csv',             parse_dates=['date'])
holidays     = pd.read_csv('holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('transactions.csv',    parse_dates=['date'])

# Preserve raw sales history for test lag computation later
sales_history = train[['date', 'store_nbr', 'family', 'sales']].copy()

print('Train:', train.shape)
print('Test: ', test.shape)
print('Date range:', train['date'].min().date(), '→', train['date'].max().date())
train.head()

In [ ]:
# Reindex to cover all dates (weekends/holidays have no price), then forward-fill
oil = (oil.set_index('date')
          .reindex(pd.date_range('2013-01-01', '2017-08-31'))
          .rename_axis('date')
          .reset_index())
oil['dcoilwtico'] = oil['dcoilwtico'].ffill()
oil.rename(columns={'dcoilwtico': 'oil_price'}, inplace=True)

print('Oil NaN remaining:', oil['oil_price'].isna().sum())
oil.tail()

In [ ]:
# National holidays apply to all 54 stores; skip 'transferred' ones (rescheduled dates)
national_hol = holidays[
    (holidays['locale'] == 'National') &
    (holidays['transferred'] == False)
][['date', 'type']].drop_duplicates('date')

national_hol.rename(columns={'type': 'holiday_type'}, inplace=True)
national_hol['is_holiday'] = 1

print('National holidays found:', len(national_hol))
national_hol.sample(5, random_state=42)

In [ ]:
def merge_all(df):
    df = df.merge(stores,                               on='store_nbr',          how='left')
    df = df.merge(oil,                                  on='date',               how='left')
    df = df.merge(transactions,                         on=['date','store_nbr'], how='left')
    df = df.merge(national_hol[['date','is_holiday']], on='date',               how='left')
    df['is_holiday']   = df['is_holiday'].fillna(0).astype(int)
    df['transactions'] = df['transactions'].fillna(0)
    df['oil_price']    = df['oil_price'].ffill().bfill()
    return df

train = merge_all(train)
test  = merge_all(test)

print('Merged train:', train.shape)
print('Merged test: ', test.shape)
print('Columns:', train.columns.tolist())
train.head(3)

---
## 2. Feature Engineering

In [ ]:
def add_date_features(df):
    df['year']           = df['date'].dt.year
    df['month']          = df['date'].dt.month
    df['day']            = df['date'].dt.day
    df['day_of_week']    = df['date'].dt.dayofweek        # 0=Mon, 6=Sun
    df['week_of_year']   = df['date'].dt.isocalendar().week.astype(int)
    df['is_weekend']     = (df['day_of_week'] >= 5).astype(int)
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end']   = df['date'].dt.is_month_end.astype(int)
    return df

train = add_date_features(train)
test  = add_date_features(test)

print('Date features added. Shape:', train.shape)
train[['date','year','month','day','day_of_week','week_of_year','is_weekend','is_holiday']].head()

In [ ]:
# Must sort before computing lags — row order within each store-family group matters
train = train.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

def add_lag_rolling(df):
    grp = df.groupby(['store_nbr', 'family'])['sales']
    for lag in [7, 14, 28]:
        df[f'sales_lag_{lag}'] = grp.shift(lag)
    for window in [7, 14, 28]:
        df[f'sales_roll_{window}'] = grp.shift(1).rolling(window).mean().reset_index(0, drop=True)
    return df

train = add_lag_rolling(train)
train.dropna(subset=['sales_lag_7', 'sales_lag_14', 'sales_lag_28'], inplace=True)

print('Shape after dropping lag NaNs:', train.shape)
lag_cols = [c for c in train.columns if 'lag' in c or 'roll' in c]
train[['date', 'store_nbr', 'family', 'sales'] + lag_cols].head()

In [ ]:
cat_cols = ['family', 'city', 'state', 'type']
le = LabelEncoder()
for col in cat_cols:
    combined_vals = pd.concat([train[col], test[col]], ignore_index=True).astype(str)
    le.fit(combined_vals)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))

feature_cols = [
    'store_nbr', 'family', 'onpromotion', 'cluster',
    'oil_price', 'transactions', 'is_holiday',
    'city', 'state', 'type',
    'year', 'month', 'day', 'day_of_week', 'week_of_year',
    'is_weekend', 'is_month_start', 'is_month_end',
    'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
    'sales_roll_7', 'sales_roll_14', 'sales_roll_28'
]

X = train[feature_cols]
y = train['sales'].clip(lower=0)

print('Feature matrix shape:', X.shape)
print(f'Target — min: {y.min():.2f}, max: {y.max():.2f}, mean: {y.mean():.2f}')
print('Features:', feature_cols)

In [ ]:
val_cutoff = train['date'].max() - pd.Timedelta(days=60)

mask_train = train['date'] <= val_cutoff
mask_val   = train['date'] >  val_cutoff

X_train, y_train = X[mask_train], y[mask_train]
X_val,   y_val   = X[mask_val],   y[mask_val]

print(f'Train: {X_train.shape[0]:,} rows  '
      f'({train.loc[mask_train,"date"].min().date()} → {train.loc[mask_train,"date"].max().date()})')
print(f'Val:   {X_val.shape[0]:,} rows  '
      f'({train.loc[mask_val,"date"].min().date()} → {train.loc[mask_val,"date"].max().date()})')

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators     = 1000,
    learning_rate    = 0.05,
    num_leaves       = 64,
    max_depth        = -1,
    min_child_samples= 20,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    random_state     = 42,
    n_jobs           = -1
)

model.fit(
    X_train, y_train,
    eval_set  = [(X_val, y_val)],
    callbacks = [lgb.early_stopping(50), lgb.log_evaluation(100)]
)

print(f'\nBest iteration: {model.best_iteration_}')

In [ ]:
y_pred_val = model.predict(X_val).clip(0)
rmsle = np.sqrt(mean_squared_log_error(y_val.clip(0), y_pred_val))

print(f'Validation RMSLE: {rmsle:.4f}')
print('(Lower is better | Kaggle competition metric)')

imp_df = pd.DataFrame({
    'feature':    feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 8))
plt.barh(range(len(imp_df)), imp_df['importance'])
plt.yticks(range(len(imp_df)), imp_df['feature'])
plt.xlabel('Feature Importance (split count)')
plt.title('LightGBM Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(imp_df.head(10).to_string(index=False))